# LeetCode 3609 — Minimum Moves to Reach Target in Grid bằng Python

- Link: https://leetcode.com/problems/minimum-moves-to-reach-target-in-grid/
- Độ khó: **Hard**
- Chủ đề: **Đi ngược từ đích** (reverse simulation), tính duy nhất của bước trước, phân tích trường hợp, giống thuật toán Euclid (LC 780 *Reaching Points*)

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minMoves(self, sx: int, sy: int, tx: int, ty: int) -> int:
```

## 1. Mục tiêu học

1. Nhận ra đi **xuôi** có hai lựa chọn mỗi bước ⇒ cây trạng thái bùng nổ; đi **ngược** thì bước trước (gần như) **duy nhất**.
2. Phân tích bước trước của `(x, y)` khi `x > y`: nếu `x ≥ 2y` thì phải là `(x/2, y)` (cần `x` chẵn); nếu `y < x < 2y` thì là `(x − y, y)`.
3. Trường hợp đặc biệt `x = y > 0`: có **hai** bước trước `(0, y)` và `(x, 0)`; `(0, 0)` không di chuyển được.
4. Số bước đi ngược là `O(log max)` (giống Euclid / chia đôi).
5. Kiểm chứng bằng BFS xuôi trên lưới nhỏ.

## 2. Đề bài & ví dụ

Từ `(x, y)`, với `m = max(x, y)`, đi tới `(x + m, y)` hoặc `(x, y + m)`. Tìm số bước ít nhất từ `(sx, sy)` tới `(tx, ty)`, hoặc `-1`.

**Ví dụ 1:** `(1, 2) → (5, 4)` → `2`.

**Ví dụ 2:** `(0, 1) → (2, 3)` → `3`.

**Ví dụ 3:** `(1, 1) → (2, 2)` → `-1`.

**Ràng buộc:** `0 ≤ sx ≤ tx ≤ 10^9`, `0 ≤ sy ≤ ty ≤ 10^9`.

In [1]:
from collections import deque
from functools import lru_cache
import random, time

examples = [((1, 2, 5, 4), 2), ((0, 1, 2, 3), 3), ((1, 1, 2, 2), -1)]
for args, exp in examples:
    print(args, "-> kỳ vọng", exp)

(1, 2, 5, 4) -> kỳ vọng 2
(0, 1, 2, 3) -> kỳ vọng 3
(1, 1, 2, 2) -> kỳ vọng -1


## 3. Kiến thức nền

### 3.1. Bước xuôi luôn làm tăng

Mỗi bước cộng `m = max(x, y) ≥ 0` vào một toạ độ; nếu `m > 0` thì toạ độ đó tăng thật sự. Từ `(0, 0)`, `m = 0` ⇒ đứng yên mãi. Vì toạ độ không giảm, có thể **cắt** mọi trạng thái vượt `(tx, ty)`.

### 3.2. Bước trước khi `x > y`

Bước cuối cộng vào `x` (vì nếu cộng vào `y` thì `y_mới = y' + m ≥ m ≥ x'`… và `x` không đổi — xét kỹ ở mục 6). Gọi trước là `(x', y)`, `m = max(x', y)`:

- nếu `x' ≥ y`: `x = 2x'` ⇒ `x' = x/2`, cần `x` chẵn và `x/2 ≥ y` ⇔ `x ≥ 2y`;
- nếu `x' < y`: `x = x' + y` ⇒ `x' = x − y`, cần `x − y < y` ⇔ `x < 2y`.

Hai điều kiện loại trừ nhau ⇒ bước trước **duy nhất**.

In [2]:
def predecessors(x, y):
    res = []
    for px in range(0, x + 1):                   # thử mọi bước trước có thể (dùng để kiểm tra lý thuyết)
        if px + max(px, y) == x and (px, y) != (x, y): res.append((px, y))
    for py in range(0, y + 1):
        if py + max(x, py) == y and (x, py) != (x, y): res.append((x, py))
    return res

for p in [(5, 4), (8, 3), (7, 3), (4, 4), (6, 0)]:
    print(p, "<-", predecessors(*p))

(5, 4) <- [(1, 4)]
(8, 3) <- [(4, 3)]
(7, 3) <- []
(4, 4) <- [(0, 4), (4, 0)]
(6, 0) <- [(3, 0)]


### 3.3. Trường hợp `x = y`

`(x, x)` với `x > 0` đến từ `(0, x)` (cộng `x` vào hoành độ) hoặc `(x, 0)`. Đây là chỗ **duy nhất** có hai nhánh — và sau đó mỗi nhánh có một toạ độ bằng 0, từ đó bước trước lại duy nhất (chia đôi toạ độ còn lại).

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Đi ngược từ `(tx, ty)`:

- Nếu đã bằng `(sx, sy)` ⇒ trả số bước.
- Nếu một toạ độ nhỏ hơn toạ độ xuất phát ⇒ `-1` (toạ độ chỉ tăng khi đi xuôi).
- `x > y`: `x ≥ 2y` ⇒ cần `x` chẵn, `x ← x/2`; ngược lại `x ← x − y`. Đối xứng khi `y > x`.
- `x = y > 0`: thử cả hai nhánh `(0, y)` và `(x, 0)`, lấy nhánh tốt hơn.

Ví dụ 1 ngược: `(5, 4)`: `4 < 5 < 8` ⇒ `(1, 4)`; `4 ≥ 2·1` ⇒ `y = 4` chẵn ⇒ `(1, 2)` = xuất phát ⇒ 2 bước. ✔

## 5. Cách 1 — Brute Force: BFS xuôi (cắt khi vượt đích)

In [3]:
def brute(sx, sy, tx, ty):
    d = {(sx, sy): 0}; q = deque([(sx, sy)])
    while q:
        x, y = q.popleft()
        if (x, y) == (tx, ty): return d[(x, y)]
        m = max(x, y)
        for nx, ny in ((x + m, y), (x, y + m)):
            if nx <= tx and ny <= ty and (nx, ny) not in d:
                d[(nx, ny)] = d[(x, y)] + 1; q.append((nx, ny))
    return -1

for args, exp in examples:
    print(brute(*args), "kỳ vọng", exp)

2 kỳ vọng 2
3 kỳ vọng 3
-1 kỳ vọng -1


**Nút thắt:** số trạng thái có thể rất lớn khi toạ độ tới `10^9`.

## 6. Ý tưởng thuật toán / bước đột phá

**Vì sao bước cuối cộng vào toạ độ lớn hơn?** Nếu đích `(x, y)` với `x > y` và bước cuối cộng vào `y`: trước đó `(x, y')`, `y = y' + max(x, y') ≥ x` — mâu thuẫn với `y < x`. Vậy bước cuối cộng vào `x`, và bước trước được xác định duy nhất (mục 3.2).

Mỗi bước ngược hoặc chia đôi một toạ độ, hoặc trừ toạ độ nhỏ khỏi toạ độ lớn khi lớn `< 2·`nhỏ — sau bước trừ, vai trò đổi và bước kế thường là chia đôi ⇒ `O(log)` bước.

## 7. Các cách tiếp theo

- **Cách 2:** đệ quy ngược có nhớ (`lru_cache`), xử lý nhánh `x = y` tự nhiên.
- **Cách 3:** vòng lặp ngược (lời giải nộp) — chỉ đệ quy một lần ở nhánh `x = y`.
- **Cách 4:** vòng lặp ngược tăng tốc: gộp nhiều lần chia đôi liên tiếp bằng phép dịch bit (bỏ số 0 cuối) khi toạ độ lớn ≥ 2 lần toạ độ nhỏ.

### 7.1. Cách 2 — Đệ quy ngược có nhớ

In [4]:
def memo_reverse(sx, sy, tx, ty):
    @lru_cache(None)
    def f(x, y):
        if (x, y) == (sx, sy): return 0
        if x < sx or y < sy or (x == 0 and y == 0): return -1
        cands = []
        if x == y:
            cands = [(0, y), (x, 0)]
        elif x > y:
            if x >= 2 * y:
                if x % 2 == 0: cands = [(x // 2, y)]
            else: cands = [(x - y, y)]
        else:
            if y >= 2 * x:
                if y % 2 == 0: cands = [(x, y // 2)]
            else: cands = [(x, y - x)]
        best = -1
        for c in cands:
            r = f(*c)
            if r != -1 and (best == -1 or r + 1 < best): best = r + 1
        return best
    return f(tx, ty)

for args, exp in examples:
    print(memo_reverse(*args), "kỳ vọng", exp)

2 kỳ vọng 2
3 kỳ vọng 3
-1 kỳ vọng -1


### 7.2. Cách 3 — Vòng lặp ngược (lời giải chính)

In [5]:
def reverse_loop(sx, sy, x, y):
    steps = 0
    while True:
        if (x, y) == (sx, sy): return steps
        if x < sx or y < sy: return -1
        if x == y:
            if x == 0: return -1
            best = -1
            for px, py in ((0, y), (x, 0)):
                r = reverse_loop(sx, sy, px, py)
                if r != -1 and (best == -1 or r + 1 < best): best = r + 1
            return -1 if best == -1 else steps + best
        if x > y:
            if x >= 2 * y:
                if x & 1: return -1
                x >>= 1
            else:
                x -= y
        else:
            if y >= 2 * x:
                if y & 1: return -1
                y >>= 1
            else:
                y -= x
        steps += 1

for args, exp in examples:
    print(reverse_loop(*args), "kỳ vọng", exp)

2 kỳ vọng 2
3 kỳ vọng 3
-1 kỳ vọng -1


### 7.3. Cách 4 — Gộp các lần chia đôi

In [6]:
def reverse_fast(sx, sy, x, y):
    steps = 0
    while True:
        if (x, y) == (sx, sy): return steps
        if x < sx or y < sy: return -1
        if x == y:
            if x == 0: return -1
            r = [reverse_fast(sx, sy, 0, y), reverse_fast(sx, sy, x, 0)]
            r = [v for v in r if v != -1]
            return steps + 1 + min(r) if r else -1
        if x > y:
            if x >= 2 * y:
                # chia đôi liên tiếp khi còn x >= 2y, x chẵn, và chưa xuống dưới sx
                t = 0
                while x >= 2 * y and not (x & 1) and x // 2 >= sx and (x, y) != (sx, sy):
                    x >>= 1; t += 1
                if t == 0: return -1
                steps += t
            else:
                x -= y; steps += 1
        else:
            if y >= 2 * x:
                t = 0
                while y >= 2 * x and not (y & 1) and y // 2 >= sy and (x, y) != (sx, sy):
                    y >>= 1; t += 1
                if t == 0: return -1
                steps += t
            else:
                y -= x; steps += 1

for args, exp in examples:
    print(reverse_fast(*args), "kỳ vọng", exp)

2 kỳ vọng 2
3 kỳ vọng 3
-1 kỳ vọng -1


**Ghi chú:** vì mỗi lần chia đôi đã là `O(1)`, Cách 4 chỉ giảm số vòng lặp Python chứ không đổi độ phức tạp; nó minh hoạ kỹ thuật "gộp bước" của LC 780 (ở đó dùng `%` để gộp nhiều phép trừ).

## 8. TRACE từng bước

In [7]:
def trace(sx, sy, x, y):
    print(f"đi ngược từ ({x}, {y}) về ({sx}, {sy})")
    steps = 0
    while (x, y) != (sx, sy):
        if x < sx or y < sy: print("  xuống dưới điểm xuất phát -> -1"); return
        if x == y: print(f"  ({x}, {y}): x = y -> hai nhánh (0, {y}) và ({x}, 0) -> đáp án", reverse_loop(sx, sy, x, y) + steps); return
        if x > y:
            if x >= 2 * y:
                if x & 1: print(f"  ({x}, {y}): x ≥ 2y nhưng x lẻ -> -1"); return
                print(f"  ({x}, {y}): x ≥ 2y -> chia đôi x"); x //= 2
            else: print(f"  ({x}, {y}): y < x < 2y -> x -= y"); x -= y
        else:
            if y >= 2 * x:
                if y & 1: print(f"  ({x}, {y}): y ≥ 2x nhưng y lẻ -> -1"); return
                print(f"  ({x}, {y}): y ≥ 2x -> chia đôi y"); y //= 2
            else: print(f"  ({x}, {y}): x < y < 2x -> y -= x"); y -= x
        steps += 1
    print("  tới điểm xuất phát sau", steps, "bước")

trace(0, 1, 2, 3)
trace(1, 0, 12, 6)
print("brute:", brute(1, 0, 12, 6))

đi ngược từ (2, 3) về (0, 1)
  (2, 3): x < y < 2x -> y -= x
  (2, 1): x ≥ 2y -> chia đôi x
  (1, 1): x = y -> hai nhánh (0, 1) và (1, 0) -> đáp án 3
đi ngược từ (12, 6) về (1, 0)
  (12, 6): x ≥ 2y -> chia đôi x
  (6, 6): x = y -> hai nhánh (0, 6) và (6, 0) -> đáp án 0
brute: -1


## 9. Code nộp LeetCode

In [8]:
class Solution:
    def minMoves(self, sx: int, sy: int, tx: int, ty: int) -> int:
        def solve(x, y):
            steps = 0
            while True:
                if x == sx and y == sy:
                    return steps
                if x < sx or y < sy:
                    return -1
                if x == y:
                    if x == 0:
                        return -1
                    best = -1
                    for px, py in ((0, y), (x, 0)):
                        r = solve(px, py)
                        if r != -1 and (best == -1 or r + 1 < best):
                            best = r + 1
                    return -1 if best == -1 else steps + best
                if x > y:
                    if x >= 2 * y:
                        if x & 1:
                            return -1
                        x >>= 1
                    else:
                        x -= y
                else:
                    if y >= 2 * x:
                        if y & 1:
                            return -1
                        y >>= 1
                    else:
                        y -= x
                steps += 1
        return solve(tx, ty)

sol = Solution()
for args, exp in examples:
    got = sol.minMoves(*args)
    print(args, "->", got, "✔" if got == exp else "✘")

(1, 2, 5, 4) -> 2 ✔
(0, 1, 2, 3) -> 3 ✔
(1, 1, 2, 2) -> -1 ✔


## 10. Vì sao thuật toán đúng

1. **Bước trước duy nhất khi `x ≠ y`** (mục 3.2 và 6). Vì vậy mọi đường xuôi tới `(x, y)` đều đi qua cùng một dãy trạng thái khi đọc ngược — số bước được xác định.
2. **`x = y > 0`** có đúng hai bước trước `(0, y)` và `(x, 0)`; mỗi nhánh sau đó lại duy nhất (một toạ độ bằng 0, toạ độ kia chỉ có thể bị chia đôi). Lấy min hai nhánh.
3. **Dừng:** mỗi bước ngược giảm tổng `x + y` thật sự; dừng khi bằng điểm xuất phát, xuống dưới nó (toạ độ chỉ tăng khi đi xuôi ⇒ không thể quay lại), gặp số lẻ ở bước chia đôi (không có bước trước), hoặc `(0, 0)`.
4. Vì đường ngược là duy nhất (trừ một lần rẽ nhánh), số bước tìm được chính là số bước **tối thiểu**.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. BFS xuôi | `O(số trạng thái ≤ đích)` | như trên |
| 2. Đệ quy ngược có nhớ | `O(log max)` | `O(log max)` |
| 3. Vòng lặp ngược | `O(log max)` | `O(1)` (một lần đệ quy) |
| 4. Gộp chia đôi | `O(log max)` | `O(1)` |

## 12. Bộ test

In [9]:
METHODS = [brute, memo_reverse, reverse_loop, reverse_fast, lambda a, b, c, d: Solution().minMoves(a, b, c, d)]
for args, exp in examples:
    assert all(f(*args) == exp for f in METHODS)
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [10]:
# 12.2 ca biên
assert Solution().minMoves(0, 0, 0, 0) == 0
assert Solution().minMoves(0, 0, 1, 0) == -1                 # (0,0) không đi được
assert Solution().minMoves(3, 5, 3, 5) == 0
assert Solution().minMoves(1, 0, 1, 1) == 1                  # (1,0) -> (1,1)
assert Solution().minMoves(0, 1, 1, 1) == 1
print("Ca biên: OK")

Ca biên: OK


In [11]:
# 12.3 ngẫu nhiên: so với BFS xuôi
rng = random.Random(3609)
for _ in range(20000):
    sx, sy = rng.randint(0, 6), rng.randint(0, 6); tx, ty = rng.randint(sx, 70), rng.randint(sy, 70)
    b = brute(sx, sy, tx, ty)
    assert all(f(sx, sy, tx, ty) == b for f in METHODS[1:]), (sx, sy, tx, ty)
print("20000 test ngẫu nhiên: OK")

20000 test ngẫu nhiên: OK


In [12]:
# 12.4 toạ độ lớn: đi xuôi ngẫu nhiên rồi kiểm tra đáp án ≤ số bước đã đi
rng = random.Random(1)
for _ in range(2000):
    sx, sy = rng.randint(0, 50), rng.randint(0, 50)
    if sx == sy == 0: continue
    x, y, steps = sx, sy, 0
    while True:
        m = max(x, y); nx, ny = (x + m, y) if rng.random() < 0.5 else (x, y + m)
        if nx > 10**9 or ny > 10**9: break
        x, y, steps = nx, ny, steps + 1
    r = Solution().minMoves(sx, sy, x, y)
    assert r != -1 and r <= steps and r == reverse_fast(sx, sy, x, y)
t0 = time.time()
for _ in range(10**4): Solution().minMoves(1, 2, 10**9, 10**9 - 7)
print("2000 hành trình ngẫu nhiên OK | 10^4 truy vấn lớn:", round(time.time() - t0, 2), "s")

2000 hành trình ngẫu nhiên OK | 10^4 truy vấn lớn: 0.01 s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Luôn trừ toạ độ nhỏ khỏi toạ độ lớn (như LC 780)

Ở bài này, khi `x ≥ 2y`, bước trước là chia đôi chứ không phải trừ `y`.

In [13]:
def wrong_euclid(sx, sy, x, y):
    steps = 0
    while (x, y) != (sx, sy):
        if x < sx or y < sy or x == y: return -1
        if x > y: x -= y                                                   # SAI
        else: y -= x
        steps += 1
    return steps

print("sai:", wrong_euclid(1, 2, 1, 4), "| đúng:", brute(1, 2, 1, 4))

sai: 2 | đúng: 1


### Lỗi 2 — Kết luận `-1` ngay khi `x = y` (quên hai nhánh `(0, y)`, `(x, 0)`)

In [14]:
def wrong_tie(sx, sy, x, y):
    steps = 0
    while (x, y) != (sx, sy):
        if x < sx or y < sy or x == y: return -1                           # SAI
        if x > y:
            if x >= 2 * y:
                if x & 1: return -1
                x >>= 1
            else: x -= y
        else:
            if y >= 2 * x:
                if y & 1: return -1
                y >>= 1
            else: y -= x
        steps += 1
    return steps

print("sai:", wrong_tie(1, 0, 1, 1), "| đúng:", brute(1, 0, 1, 1))

sai: -1 | đúng: 1


### Lỗi 3 — Không kiểm tra `x` lẻ khi chia đôi (dùng `//` làm tròn)

In [15]:
def wrong_odd(sx, sy, x, y):
    steps = 0
    while (x, y) != (sx, sy):
        if x < sx or y < sy: return -1
        if x == y: return reverse_loop(sx, sy, x, y) + steps if reverse_loop(sx, sy, x, y) != -1 else -1
        if x > y: x = x // 2 if x >= 2 * y else x - y                        # SAI: không kiểm tra lẻ
        else: y = y // 2 if y >= 2 * x else y - x
        steps += 1
    return steps

rng = random.Random(31)
while True:
    a = (rng.randint(0, 4), rng.randint(0, 4)); b = (rng.randint(a[0], 40), rng.randint(a[1], 40))
    if wrong_odd(*a, *b) != brute(*a, *b): break
print("phản ví dụ:", a, "->", b, "| sai:", wrong_odd(*a, *b), "| đúng:", brute(*a, *b))

phản ví dụ: (1, 0) -> (9, 7) | sai: 5 | đúng: -1


### Lỗi 4 — Coi `(0, 0)` đi được (vòng lặp vô hạn khi BFS xuôi)

Từ `(0, 0)`, `m = 0` ⇒ cả hai bước giữ nguyên trạng thái. BFS có tập `visited` thì không treo, nhưng mô phỏng "đi tới khi đạt" sẽ lặp vô hạn. Xử lý riêng: `(0, 0)` chỉ đạt chính nó.

In [16]:
print("(0,0) -> (0,0):", Solution().minMoves(0, 0, 0, 0), "| (0,0) -> (0,5):", Solution().minMoves(0, 0, 0, 5))

(0,0) -> (0,0): 0 | (0,0) -> (0,5): -1


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Bước xuôi có nhiều lựa chọn nhưng làm toạ độ **tăng nhanh** ⇒ thử đi **ngược**: thường bước trước duy nhất.
- Xác định bước cuối bằng cách so sánh các toạ độ (toạ độ lớn hơn là toạ độ vừa được cộng).
- Phân tích đủ các ca biên: bằng nhau, bằng 0, lẻ/chẵn.
- So sánh với bài quen (LC 780) nhưng cẩn thận khác biệt luật di chuyển.

## 15. Biến thể của bài

1. **LC 780 *Reaching Points***: `(x, y) → (x + y, y)` hoặc `(x, x + y)` — ngược bằng `%`.
2. **Cộng `min(x, y)`** thay vì `max`: bước trước có thể không duy nhất — cần phân tích lại.
3. **Đếm số cách** đi tới đích: bằng 1 (hoặc 2 nếu đi qua nhánh `x = y`) nhờ tính duy nhất.
4. **Ba toạ độ** `(x, y, z)` cộng `max` vào một toạ độ: tổng quát hoá phân tích bước trước.

In [17]:
# Biến thể 1: LC 780 reaching points (đi ngược bằng %)
def reaching_points(sx, sy, tx, ty):
    while tx >= sx and ty >= sy:
        if tx == ty: break
        if tx > ty:
            if ty > sy: tx %= ty
            else: return (tx - sx) % ty == 0
        else:
            if tx > sx: ty %= tx
            else: return (ty - sy) % tx == 0
    return tx == sx and ty == sy

def brute_780(sx, sy, tx, ty):
    seen = {(sx, sy)}; st = [(sx, sy)]
    while st:
        x, y = st.pop()
        if (x, y) == (tx, ty): return True
        for nx, ny in ((x + y, y), (x, x + y)):
            if nx <= tx and ny <= ty and (nx, ny) not in seen: seen.add((nx, ny)); st.append((nx, ny))
    return False

rng = random.Random(15)
for _ in range(3000):
    sx, sy = rng.randint(1, 6), rng.randint(1, 6); tx, ty = rng.randint(sx, 40), rng.randint(sy, 40)
    assert reaching_points(sx, sy, tx, ty) == brute_780(sx, sy, tx, ty)
print("Biến thể 1 (LC 780) OK")

Biến thể 1 (LC 780) OK


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Khi nào dùng |
|---|---|---|---|
| 1. BFS xuôi | Duyệt trạng thái | lớn | Kiểm chứng |
| 2. Đệ quy ngược có nhớ | Bước trước + nhánh `x = y` | `O(log)` | Dễ viết đúng |
| 3. Vòng lặp ngược | Như 2, lặp | `O(log)` | **Lời giải chuẩn** |
| 4. Gộp chia đôi | Giảm số vòng lặp | `O(log)` | Minh hoạ gộp bước |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `minMoves(2, 1, 8, 1)`.

**Bài 2 (tính tay).** Đi ngược `(1, 3) → (7, 6)`? Ghi từng bước.

**Bài 3 (điền TODO).** Viết `path(sx, sy, tx, ty)` trả về dãy trạng thái (xuôi) của đường ngắn nhất.

**Bài 4 (tự viết lại).** Viết `reachable_targets(sx, sy, limit)`: mọi `(x, y)` với `x, y ≤ limit` đạt được, và đếm.

**Bài 5 (mini challenge).** Biến thể cộng `min(x, y)`: viết BFS và tìm ví dụ có hai bước trước khác nhau (chứng tỏ đi ngược không còn duy nhất).

In [18]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def path(sx, sy, tx, ty):
    # TODO
    pass

In [19]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def min_variant_bfs(sx, sy, tx, ty):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `(8, 1)`: `8 ≥ 2` ⇒ `(4, 1)` ⇒ `(2, 1)`. (2) 2 bước. (3) `2`.

**Bài 2.** (1) `(7, 6)`: `6 < 7 < 12` ⇒ `(1, 6)`. (2) `6 ≥ 2` ⇒ `(1, 3)`. (3) `2`.

**Bài 3.** (1) Đi ngược ghi lại trạng thái. (2) Ở nhánh `x = y` chọn nhánh cho kết quả nhỏ hơn. (3) Đảo danh sách.

**Bài 4.** (1) BFS xuôi từ `(sx, sy)` cắt ở `limit`. (2) Đếm số trạng thái. (3) Đối chiếu từng điểm bằng lời giải.

**Bài 5.** (1) BFS với `m = min(x, y)`. (2) Với mỗi trạng thái, liệt kê bước trước. (3) Tìm điểm có ≥ 2 bước trước.

## 19. Đáp án tham khảo cho bài tập

In [20]:
# Bài 1, 2
assert Solution().minMoves(2, 1, 8, 1) == 2 == brute(2, 1, 8, 1)
assert Solution().minMoves(1, 3, 7, 6) == 2 == brute(1, 3, 7, 6)
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [21]:
# Bài 3, 4
def path_sol(sx, sy, tx, ty):
    if Solution().minMoves(sx, sy, tx, ty) == -1: return None
    x, y = tx, ty; out = [(x, y)]
    while (x, y) != (sx, sy):
        if x == y:
            a, b = reverse_loop(sx, sy, 0, y), reverse_loop(sx, sy, x, 0)
            x, y = ((0, y) if a != -1 and (b == -1 or a <= b) else (x, 0))
        elif x > y: x = x // 2 if x >= 2 * y else x - y
        else: y = y // 2 if y >= 2 * x else y - x
        out.append((x, y))
    return out[::-1]

def reachable_targets_sol(sx, sy, limit):
    seen = {(sx, sy)}; q = deque([(sx, sy)])
    while q:
        x, y = q.popleft(); m = max(x, y)
        for nx, ny in ((x + m, y), (x, y + m)):
            if nx <= limit and ny <= limit and (nx, ny) not in seen: seen.add((nx, ny)); q.append((nx, ny))
    return seen

rng = random.Random(4)
for _ in range(500):
    sx, sy = rng.randint(0, 5), rng.randint(0, 5); tx, ty = rng.randint(sx, 60), rng.randint(sy, 60)
    p = path_sol(sx, sy, tx, ty); b = brute(sx, sy, tx, ty)
    if p is None: assert b == -1; continue
    assert len(p) - 1 == b and p[0] == (sx, sy) and p[-1] == (tx, ty)
    for (x1, y1), (x2, y2) in zip(p, p[1:]):
        m = max(x1, y1); assert (x2, y2) in ((x1 + m, y1), (x1, y1 + m))
R = reachable_targets_sol(1, 2, 60)
assert all(Solution().minMoves(1, 2, x, y) != -1 for x, y in R)
print("Bài 3, 4 OK; ví dụ 2:", path_sol(0, 1, 2, 3), "| số điểm đạt được từ (1,2) trong [0,60]^2:", len(R))

Bài 3, 4 OK; ví dụ 2: [(0, 1), (1, 1), (2, 1), (2, 3)] | số điểm đạt được từ (1,2) trong [0,60]^2: 126


In [22]:
# Bài 5: cộng min(x, y) — bước trước không còn duy nhất
def min_variant_bfs_sol(sx, sy, tx, ty):
    d = {(sx, sy): 0}; q = deque([(sx, sy)])
    while q:
        x, y = q.popleft()
        if (x, y) == (tx, ty): return d[(x, y)]
        m = min(x, y)
        for nx, ny in ((x + m, y), (x, y + m)):
            if nx <= tx and ny <= ty and (nx, ny) not in d: d[(nx, ny)] = d[(x, y)] + 1; q.append((nx, ny))
    return -1

def preds_min(x, y, limit=60):
    return [(px, py) for px in range(limit) for py in range(limit)
            if ((px + min(px, py), py) == (x, y) or (px, py + min(px, py)) == (x, y)) and (px, py) != (x, y)]

ex = next((x, y) for x in range(1, 30) for y in range(1, 30) if len(preds_min(x, y)) >= 2)
print("Bài 5: điểm", ex, "có các bước trước", preds_min(*ex), "| BFS (1,1) ->", ex, ":", min_variant_bfs_sol(1, 1, *ex))

Bài 5: điểm (2, 2) có các bước trước [(1, 2), (2, 1)] | BFS (1,1) -> (2, 2) : 2


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
đi ngược (x, y) từ đích:
    == start -> số bước; < start (một toạ độ) -> -1
    x == y: 0 -> -1; ngược lại min(solve(0, y), solve(x, 0)) + 1
    x > y:  x >= 2y -> (x lẻ ? -1 : x //= 2);  else x -= y
    y > x:  đối xứng
```

**Checklist:**

- [ ] Phân biệt `x ≥ 2y` (chia đôi) và `y < x < 2y` (trừ)?
- [ ] Kiểm tra số lẻ khi chia đôi?
- [ ] Hai nhánh khi `x = y > 0`?
- [ ] `(0, 0)` không đi được?
- [ ] Đã so với BFS xuôi?